# 02 — Preprocesamiento y Feature Engineering
**CRISP-DM: Preparación de los datos** · Etapa 1: *Preprocesamiento y feature engineering*

Limpieza, tratamiento de outliers, resampleo semanal, relleno de huecos,
integración de las 4 fuentes externas y creación de características.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))  # importar el paquete sin instalar
import pandas as pd
from prediccion_precios import data_loading as dl, preprocessing as pp, features as ft, eda, config
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)

## 1. Precios objetivo → limpieza → outliers

In [ ]:
precios = dl.filtrar_productos_objetivo(dl.cargar_precios_consolidados())
precios = pp.limpiar_precios(precios)
precios = pp.tratar_outliers(precios)      # winsorización IQR por producto
precios.shape

## 2. Resampleo semanal + rejilla continua (interpolación)

In [ ]:
sem = pp.resamplear_frecuencia(precios)     # media semanal por producto
sem = pp.tratar_faltantes(sem)             # reindexa y interpola huecos
print(sem.shape, '| NaN precio:', int(sem['Precio'].isna().sum()))

## 3. Integración de fuentes externas
combustible (diario), precipitación (diario), IPC transporte (mensual), producción FAOSTAT (anual).

In [ ]:
comb = dl.cargar_combustible()
prec = dl.cargar_precipitacion()
ipc  = dl.cargar_ipc()
prod = dl.cargar_produccion()
data = pp.integrar_fuentes(sem, combustible=comb, precipitacion=prec, ipc=ipc, produccion=prod)
print(data.shape); data.head(3)

## 4. Feature engineering
calendario cíclico, lags, medias/desv móviles y temporada de cosecha (aporte local).

In [ ]:
data = ft.agregar_variables_calendario(data)
data = ft.agregar_lags(data)
data = ft.agregar_medias_moviles(data)
data = ft.marcar_temporada_cosecha(data)
print(data.shape); list(data.columns)

## 5. Guardar dataset de modelado

In [ ]:
ruta = pp.guardar_processed(data)
print('Guardado en', ruta)